In [ ]:
!pip install -q vllm pandas tqdm

In [ ]:
import time

import torch
import pandas as pd
from tqdm.auto import tqdm

from vllm import LLM, SamplingParams

from google.colab import drive

In [ ]:
drive.mount('/content/drive')

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/balanced_lit.csv")
df = df[df['author'] == 'Dostoyevsky']
df.head()

In [ ]:
df = df.sample(1000)
df.head()
df

In [ ]:
MODEL_NAME = "Qwen/Qwen2-7B-Instruct"

llm = LLM(
    model=MODEL_NAME,
    dtype="float16",
    trust_remote_code=True,
    gpu_memory_utilization=0.95
)


In [ ]:
torch.cuda.empty_cache()

In [ ]:
sampling_params = SamplingParams(
    temperature=0,
    top_p=1.0,
    max_tokens=256
)

In [ ]:
def build_prompt(text):
    return f"""
      Перепиши текст в нейтральном стиле.

      Требования:
      - сохранить смысл
      - убрать авторскую стилистику
      - убрать метафоры, эмоциональность
      - сделать обычный современный текст
      - вернуть только результат

      Текст:
      {text}

      Нейтральный вариант:
      """

In [ ]:
BATCH_SIZE = 8
results = []

texts = df["text"].tolist()
authors = df["author"].tolist()

In [ ]:
for i in tqdm(range(0, len(df), BATCH_SIZE)):

    batch_texts = texts[i:i+BATCH_SIZE]
    batch_authors = authors[i:i+BATCH_SIZE]

    prompts = [build_prompt(t) for t in batch_texts]

    outputs = llm.generate(prompts, sampling_params)

    for j, out in enumerate(outputs):
        neutral = out.outputs[0].text.strip()

        results.append({
            "author": batch_authors[j],
            "styled_text": batch_texts[j],
            "neutral_text": neutral
        })


out_df = pd.DataFrame(results)


In [ ]:
print(out_df.head())

In [ ]:
print(out_df.iloc[4]['styled_text'])
print("===")
print(out_df.iloc[4]['neutral_text'])


In [ ]:
out_df.to_csv("/content/drive/MyDrive/parallel_1000_dostoyesky.csv", index=False, encoding="utf-8")